# Google Earth Engine Tutorial

This tutorial is borrowed from NCEAS: https://learning.nceas.ucsb.edu/2022-09-arctic/sections/15-google-earth-engine.html.

GEE Data catalogue is here: https://developers.google.com/earth-engine/datasets

More tutorial here: https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api

In [1]:
# Dependencies
import ee
import geemap

In [2]:
# triggers the authentication process
ee.Authenticate() 

True

In [3]:
# Initialize
ee.Initialize() 

In [4]:
# Create a leaflet map
myMap = geemap.Map(center = [40, -100], zoom = 2)
myMap

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [5]:
# Gather ECMWF ERA5 Weather data collection
weatherData = ee.ImageCollection('ECMWF/ERA5/DAILY')

In [6]:
# Check it out
print(type(weatherData))
weatherData

<class 'ee.imagecollection.ImageCollection'>


In [7]:
# select desired bands (total_preciptation)
precip = weatherData.select("total_precipitation")
print(type(weatherData))

<class 'ee.imagecollection.ImageCollection'>


In [12]:
# initial date of interest (inclusive)
i_date = '1995-01-01'

# final data of interest (exclusive)
f_date = '1996-01-01'

# select desired bands (total_preciptation), dates, and calculate mean precipitation across that date range
precip = weatherData.select("total_precipitation").filterDate(i_date, f_date).mean()

In [13]:
# Check it out
print(type(precip))
precip

<class 'ee.image.Image'>


In [14]:
# Make a color palet
precip_palette = {
    'min':0,
    'max':0.01,
    'palette': ['#FFFFFF', '#00FFFF', '#0080FF', '#DA00FF', '#FFA400', '#FF0000']
}

In [15]:
# Plot it
myMap.addLayer(precip, precip_palette, 'total precipitation', opacity = 0.3)
myMap

Map(bottom=1474.0, center=[-25.48295117535532, 18.808593750000004], controls=(WidgetControl(options=['position…

In [16]:
from IPython.display import Image

# Define the urban location of interest as a point near Lyon, France.
u_lon = 4.8148
u_lat = 45.7758
u_poi = ee.Geometry.Point(u_lon, u_lat)

# Define a region of interest with a buffer zone of 1000 km around Lyon.
roi = u_poi.buffer(1e6)

# Get a feature collection of administrative boundaries.
countries = ee.FeatureCollection('FAO/GAUL/2015/level0').select('ADM0_NAME')

# Filter the feature collection to subset France.
fr = countries.filter(ee.Filter.eq('ADM0_NAME', 'France'))

# Clip the image by France.
precip_fr = precip.clip(fr)

# Create the URL associated with the styled image data.
url = precip_fr.getThumbUrl({
    'min': 0, 'max': 0.01, 'region': roi, 'dimensions': 512,
    'palette': ['#FFFFFF', '#00FFFF', '#0080FF', '#DA00FF', '#FFA400', '#FF0000']})

# Display a thumbnail of elevation in France.
Image(url=url)

In [17]:
# Get a link to download 
link = precip_fr.getDownloadURL({
    'scale': 30,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': u_poi})
print(link)

https://earthengine.googleapis.com/v1/projects/earthengine-legacy/thumbnails/22bf1f454583dc4c995df01e7ce0ed93-8724871955dbd1767f693f8ea996f246:getPixels
